<a href="https://colab.research.google.com/github/e-saidha/skyhack_querykings/blob/main/Difficulty_Score_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
flights = pd.read_csv('/content/Flight Level Data.csv')
pnr_flight = pd.read_csv('/content/PNR+Flight+Level+Data.csv')
pnr_rem = pd.read_csv('/content/PNR Remark Level Data.csv')
bag_level = pd.read_csv('/content/Bag+Level+Data.csv')
airports = pd.read_csv('/content/Airports Data.csv')
airports = airports.drop_duplicates()

In [2]:
import duckdb, pandas as pd, numpy as np
con = duckdb.connect()


con.register("flights", flights)
con.register("pnr_flight", pnr_flight)
con.register("pnr_rem", pnr_rem)
con.register("bag_level", bag_level)
con.register("airports", airports)


In [3]:

con.execute("""
CREATE OR REPLACE VIEW v_flights_ord AS
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY company_id, flight_number, scheduled_departure_date_local,
                   scheduled_departure_station_code, scheduled_arrival_station_code
      ORDER BY COALESCE(actual_departure_datetime_local, scheduled_departure_datetime_local) DESC
    ) AS rn
  FROM flights
  WHERE scheduled_departure_station_code = 'ORD'
) t
WHERE rn = 1
""")
n_flights = con.execute("SELECT COUNT(*) FROM v_flights_ord").fetchone()[0]



In [4]:
# 2a) Collapse to one row per (flight+record_locator): take the latest PNR snapshot
con.execute("""
CREATE OR REPLACE VIEW v_pnr_per_booking AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  record_locator,
  -- latest snapshot per booking (handle updates)
  MAX(COALESCE(total_pax,0))                                              AS pnr_pax,
  MAX(COALESCE(lap_child_count,0))                                        AS pnr_lap_childs,
  MAX(COALESCE(CAST(basic_economy_ind AS BIGINT),0))                      AS pnr_basic_econ,
  MAX(
    CASE
      WHEN UPPER(TRIM(CAST(is_stroller_user AS VARCHAR))) IN ('Y','YES','TRUE','1') THEN 1
      ELSE 0
    END
  )                                                                        AS pnr_stroller_user
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY company_id, flight_number, scheduled_departure_date_local,
                   scheduled_departure_station_code, scheduled_arrival_station_code,
                   record_locator
      ORDER BY pnr_creation_date DESC
    ) AS rn
  FROM pnr_flight
) q
WHERE rn = 1
GROUP BY 1,2,3,4,5,6
""")

# 2b) Aggregate per flight (sum across distinct PNRs)
con.execute("""
CREATE OR REPLACE VIEW v_pnr_agg AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  SUM(pnr_pax)           AS pax_total,
  SUM(pnr_lap_childs)    AS lap_childs,
  SUM(pnr_basic_econ)    AS basic_econ,
  SUM(pnr_stroller_user) AS stroller_users,
  COUNT(*)               AS distinct_pnrs
FROM v_pnr_per_booking
GROUP BY 1,2,3,4,5
""")


In [5]:
con.execute("""
CREATE OR REPLACE VIEW v_bags_agg AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  -- origin == checked
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(bag_type))='ORIGIN' THEN bag_tag_unique_number END)        AS checked_bags,
  -- transfer includes both 'transfer' and 'hot transfer'
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(bag_type)) IN ('TRANSFER','HOT TRANSFER')
                      THEN bag_tag_unique_number END)                                           AS transfer_bags
FROM bag_level
GROUP BY 1,2,3,4,5
""")


In [6]:
con.execute("""
CREATE OR REPLACE VIEW v_ssr_agg AS
SELECT
  p.company_id, p.flight_number, p.scheduled_departure_date_local,
  p.scheduled_departure_station_code, p.scheduled_arrival_station_code,
  COUNT(*) AS ssr_count
FROM pnr_rem r
JOIN v_pnr_per_booking p
  ON r.record_locator = p.record_locator
 AND r.flight_number  = p.flight_number
GROUP BY 1,2,3,4,5
""")


In [7]:
df_master = con.execute("""
WITH base AS (
  SELECT
    f.*,
    (f.scheduled_ground_time_minutes - f.minimum_turn_minutes) AS slack_mins,
    EXTRACT(hour FROM CAST(f.scheduled_departure_datetime_local AS TIMESTAMP)) AS dep_hour,
    EXTRACT(dow  FROM CAST(f.scheduled_departure_datetime_local AS TIMESTAMP)) AS dep_dow
  FROM v_flights_ord f
)
SELECT
  b.*,
  COALESCE(p.pax_total,0)        AS pax_total,
  COALESCE(p.lap_childs,0)       AS lap_childs,
  COALESCE(p.basic_econ,0)       AS basic_econ,
  COALESCE(p.stroller_users,0)   AS stroller_users,
  COALESCE(p.distinct_pnrs,0)    AS distinct_pnrs,
  COALESCE(bg.checked_bags,0)    AS checked_bags,
  COALESCE(bg.transfer_bags,0)   AS transfer_bags,
  CASE WHEN COALESCE(bg.checked_bags,0) > 0
       THEN CAST(bg.transfer_bags AS DOUBLE)/bg.checked_bags
       ELSE NULL END             AS transfer_ratio,
  COALESCE(s.ssr_count,0)        AS ssr_count,
  apt.iso_country_code           AS arrival_country
FROM base b
LEFT JOIN v_pnr_agg  p  USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN v_bags_agg bg USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN v_ssr_agg  s  USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN airports apt
       ON b.scheduled_arrival_station_code = apt.airport_iata_code
""").df()

print("Master rows:", len(df_master))
assert len(df_master) == n_flights, "Row-count mismatch: master != flights_ORD"


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Master rows: 8063


In [8]:
# --- Feature engineering on df_master (run once) ---
df = df_master.copy()

# delay target
if "dep_delay_min" not in df.columns:
    df["scheduled_departure_datetime_local"] = pd.to_datetime(df["scheduled_departure_datetime_local"], errors='coerce')
    df["actual_departure_datetime_local"]    = pd.to_datetime(df["actual_departure_datetime_local"], errors='coerce')
    df["dep_delay_min"] = (df["actual_departure_datetime_local"] - df["scheduled_departure_datetime_local"]).dt.total_seconds() / 60

df["is_difficult"] = (df["dep_delay_min"] > 15).astype(int)

# basic engineered features
df["load_factor"] = (df["pax_total"] / df["total_seats"]).clip(lower=0, upper=1.0)
if "slack_mins" not in df.columns:
    df["slack_mins"] = df["scheduled_ground_time_minutes"] - df["minimum_turn_minutes"]

checked  = df["checked_bags"].fillna(0)
transfer = df["transfer_bags"].fillna(0)
total_bags = checked + transfer
df["transfer_share"] = np.where(total_bags > 0, transfer / total_bags, 0.0)
df["bag_volume"]     = total_bags

pax = df["pax_total"].fillna(0)
df["ssr_per_100pax"] = np.where(pax > 0, df["ssr_count"] / pax * 100, 0.0)

# hour + cyclic encoding
if "dep_hour" not in df.columns:
    df["dep_hour"] = pd.to_datetime(df["scheduled_departure_datetime_local"], errors='coerce').dt.hour
mode_series = df["dep_hour"].mode()
df["dep_hour"] = df["dep_hour"].fillna(int(mode_series.iloc[0]) if len(mode_series) else 0).astype(int)
df["dep_hour_sin"] = np.sin(2 * np.pi * df["dep_hour"] / 24)
df["dep_hour_cos"] = np.cos(2 * np.pi * df["dep_hour"] / 24)

# categoricals
for col in ["fleet_type", "carrier", "arrival_country"]:
    df[col] = df[col].fillna("UNK").astype(str)

# ---- Define ONE canonical feature list and reuse for train + score ----
TRAIN_FEATURES_NUM = [
    "slack_mins", "load_factor", "ssr_count", "ssr_per_100pax",
    "transfer_share", "bag_volume", "dep_hour_sin", "dep_hour_cos",
    "dep_dow", "checked_bags", "transfer_bags"
]
TRAIN_FEATURES_CAT = ["fleet_type", "carrier", "arrival_country"]
TRAIN_FEATURES_ALL = TRAIN_FEATURES_NUM + TRAIN_FEATURES_CAT

model_df = df[["company_id","flight_number","scheduled_departure_date_local","dep_delay_min","is_difficult"] + TRAIN_FEATURES_ALL].copy()
model_df[TRAIN_FEATURES_NUM] = model_df[TRAIN_FEATURES_NUM].fillna(0)

print("Modeling frame shape:", model_df.shape)
print(f"Target distribution (% difficult): {model_df['is_difficult'].mean()*100:.1f}%")
display(model_df.head(3))

X = model_df.drop(columns=["company_id","flight_number","scheduled_departure_date_local","dep_delay_min","is_difficult"])
y = model_df["is_difficult"]


Modeling frame shape: (8063, 19)
Target distribution (% difficult): 26.8%


,company_id,flight_number,scheduled_departure_date_local,dep_delay_min,is_difficult,slack_mins,load_factor,ssr_count,ssr_per_100pax,transfer_share,bag_volume,dep_hour_sin,dep_hour_cos,dep_dow,checked_bags,transfer_bags,fleet_type,carrier,arrival_country
0,UA,767,2025-08-05,-1.0,0,5,0.175000,1,2.857143,0.659574,47,-7.071068e-01,0.707107,2,16,31,A321-2NX,Mainline,US
1,UA,503,2025-08-05,-2.0,0,228,0.166667,2,5.128205,0.533333,90,-5.000000e-01,-0.866025,2,42,48,B757-300,Mainline,US
2,UA,881,2025-08-05,-2.0,0,1265,0.229560,2,2.739726,0.831325,166,1.224647e-16,-1.000000,2,28,138,B787-10,Mainline,JP


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

pre = ColumnTransformer(
    [
        ("num", StandardScaler(), TRAIN_FEATURES_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), TRAIN_FEATURES_CAT),
    ],
    remainder="drop"
)



logreg = LogisticRegression(
    solver="saga",
    max_iter=5000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline([("pre", pre), ("clf", logreg)])
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
prec, rec, thr = precision_recall_curve(y_test, proba)
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = thr[np.nanargmax(f1_scores)]

print("Model training complete!")
print(f"ROC AUC: {auc:.3f}")
print(f"Best threshold (F1): {best_thr:.3f}")

y_pred = (proba >= best_thr).astype(int)
print("\nClassification Report @best F1:")
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Model training complete!
ROC AUC: 0.694
Best threshold (F1): 0.458

Classification Report @best F1:
              precision    recall  f1-score   support

           0      0.844     0.602     0.703      1771
           1      0.390     0.696     0.500       648

    accuracy                          0.627      2419
   macro avg      0.617     0.649     0.601      2419
weighted avg      0.722     0.627     0.648      2419

Confusion Matrix:
 [[1066  705]
 [ 197  451]]


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipe_rf = Pipeline(steps=[("pre", pre), ("clf", rf)])
pipe_rf.fit(X_train, y_train)

rf_proba = pipe_rf.predict_proba(X_test)[:, 1]
rf_pred  = (rf_proba >= 0.5).astype(int)

rf_auc = roc_auc_score(y_test, rf_proba)
print("Random Forest ROC AUC:", round(rf_auc, 3))
print("\nClassification Report (0.5 threshold):\n", classification_report(y_test, rf_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))

# Feature importances (numeric + expanded categorical)
ohe = pipe_rf.named_steps["pre"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(TRAIN_FEATURES_CAT)
feat_names = TRAIN_FEATURES_NUM + list(cat_feature_names)

importances = pipe_rf.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"Feature": feat_names, "Importance": importances}) \
           .sort_values("Importance", ascending=False).head(15)
print("\nTop 15 Feature Importances:")
display(imp_df)


Random Forest ROC AUC: 0.776

Classification Report (0.5 threshold):
               precision    recall  f1-score   support

           0      0.813     0.938     0.871      1771
           1      0.709     0.409     0.519       648

    accuracy                          0.797      2419
   macro avg      0.761     0.674     0.695      2419
weighted avg      0.785     0.797     0.777      2419

Confusion Matrix:
 [[1662  109]
 [ 383  265]]

Top 15 Feature Importances:


,Feature,Importance
0,slack_mins,0.239709
1,load_factor,0.132350
6,dep_hour_sin,0.111574
8,dep_dow,0.096578
7,dep_hour_cos,0.084532
3,ssr_per_100pax,0.042950
5,bag_volume,0.040905
10,transfer_bags,0.037903
4,transfer_share,0.033951
9,checked_bags,0.029537


In [12]:

for c in ["load_factor","slack_mins","transfer_share","bag_volume","ssr_per_100pax",
          "dep_hour","dep_hour_sin","dep_hour_cos"]:
    df_master[c] = df[c]


X_score = df_master[TRAIN_FEATURES_ALL]
df_master["difficulty_score"] = pipe_rf.predict_proba(X_score)[:, 1]


df_master["daily_rank"] = (
    df_master.groupby("scheduled_departure_date_local")["difficulty_score"]
    .rank(method="first", ascending=False)
)


group_sizes = df_master.groupby("scheduled_departure_date_local")["difficulty_score"].transform("size")
df_master["rank_pct"] = ((df_master["daily_rank"] - 1) / (group_sizes - 1)).fillna(0).clip(0, 1)

df_master["difficulty_class"] = pd.cut(
    df_master["rank_pct"],
    bins=[0, 0.2, 0.5, 1.0],
    labels=["Difficult", "Medium", "Easy"],
    include_lowest=True
)

submission_cols = [
    "company_id","flight_number","scheduled_departure_date_local",
    "scheduled_departure_station_code","scheduled_arrival_station_code",

    *TRAIN_FEATURES_ALL,

    "difficulty_score","daily_rank","difficulty_class"
]

submission = (
    df_master[submission_cols]
    .sort_values(["scheduled_departure_date_local","daily_rank"])
    .reset_index(drop=True)
)

out_path = "test_querykings.csv"
submission.to_csv(out_path, index=False)
print(f"Submission file created: {out_path}, shape={submission.shape}")
display(submission.sample(10))


Submission file created: test_querykings.csv, shape=(8063, 22)


,company_id,flight_number,scheduled_departure_date_local,scheduled_departure_station_code,scheduled_arrival_station_code,slack_mins,load_factor,ssr_count,ssr_per_100pax,transfer_share,...,dep_hour_cos,dep_dow,checked_bags,transfer_bags,fleet_type,carrier,arrival_country,difficulty_score,daily_rank,difficulty_class
2855,UA,1205,2025-08-06,ORD,EWR,865,0.178771,0,0.000000,0.000000,...,-9.659258e-01,3,0,0,B737-900,Mainline,US,0.496667,143.0,Medium
4309,OO,5964,2025-08-08,ORD,CID,38,0.144737,0,0.000000,0.000000,...,-2.588190e-01,5,0,0,ERJ-175,Express,US,0.003333,544.0,Easy
836,UA,987,2025-08-02,ORD,CDG,30,0.084906,0,0.000000,0.390000,...,-1.836970e-16,6,61,39,B787-10,Mainline,FR,0.093333,279.0,Easy
5564,UA,2658,2025-08-11,ORD,SMF,58,0.146667,0,0.000000,0.000000,...,-5.000000e-01,1,0,0,A320-200,Mainline,US,0.186667,177.0,Medium
6016,OO,5474,2025-08-12,ORD,MBS,283,0.300000,1,6.666667,0.000000,...,-1.836970e-16,2,0,0,CRJ-550,Express,US,0.863333,66.0,Difficult
1584,OO,5437,2025-08-03,ORD,DLH,271,0.160000,1,12.500000,0.944444,...,-1.000000e+00,0,1,17,CRJ-550,Express,US,0.020000,507.0,Easy
5428,UA,2147,2025-08-11,ORD,ALB,39,0.166667,0,0.000000,0.000000,...,-1.836970e-16,1,0,0,A320-200,Mainline,US,0.766667,41.0,Difficult
2255,UA,2347,2025-08-05,ORD,CLE,11,0.084337,0,0.000000,0.863636,...,7.071068e-01,2,3,19,B737-800,Mainline,US,0.306667,69.0,Difficult
7356,OO,5292,2025-08-14,ORD,MTY,1,0.263158,1,5.000000,0.000000,...,-5.000000e-01,4,0,0,ERJ-175,Express,MX,0.043333,410.0,Easy
2803,OO,5280,2025-08-06,ORD,DSM,31,0.105263,0,0.000000,1.000000,...,7.071068e-01,3,0,12,ERJ-175,Express,US,0.770000,91.0,Difficult
